<a href="https://colab.research.google.com/github/crreyes-unal/Reporte-problemas-MpAM/blob/main/Ejercicio6_RetoNetflix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

***Reto Netflix: A partir del artículo *KDD Cup 2007 Task 1 Winner Report* coautorado por Miklós Kurucz, responder detalladamente cómo se logró resolver el Reto Netflix a partir de la solución del problema de factorización de matrices no negativas. (Sugerencia: Hacer un paralelo de Temas vs Temáticas y de Temáticas vs Películas y deducir cómo es el comportamiento de la factorización a partir de este paralelo).***

En el artículo, se describe la solución para la Tarea 1 de la KDD Cup 2007 la cual consiste en predecir qué usuarios calificaron qué películas en 2006. Para lograr esto, no usaron un único modelo, sino una combinación de predictores. La tecnica fundamental fue la descomposición en valores singulares o por sus siglas SVD, operando sobre una matriz de valores $0$ y $1$ que representaba los pares conocidos de usuario-película.

A continuación, se explica el comportamiento de esta factorización utilizando el paralelo sugerido.

Se construye una matriz $W$, en donde las filas son usuarios y las columnas son películas con valor $1$ si se calificó y $0$ si no. La factorización descompone esta matriz en el producto de matrices más pequeñas y viene dada por

$$W = U^T \Sigma V$$

El paralelo se puede dar en base cada parte de la ecuación. Se desgloza cada una a continuación.

*   La matriz $U^T$ que corresponde a temas o usuarios vs. temáticas, la cual mapea a cada usuario con un conjunto de características ocultas las temáticas, los cuales se llaman factores latentes. Esta matriz nos puede decir, por ejemplo, qué tanta afinidad tiene un usuario por las temáticas de comedia o acción.
*   La matriz $V$ que corresponde a temáticas vs. películas, la cual mapea las temáticas con las películas específicas. En el caso propuesto, indica en qué medida una película pertenece a la temática de comedia o acción.
*   La matriz $\Sigma$, la cual es una matriz diagonal que pondera la importancia de cada una de estas temáticas en el conjunto total de datos.

Al multiplicar las matrices anteriores, el sistema predice la probabilidad de interacción basándose en la afinidad del perfil del usuario por ciertas temáticas, combinado con qué tanto las películas pertenecen a esas temáticas.

Para evitar sobreajuste, lo que en el artículo se llama overfitting, el equipo no usó todas las temáticas posibles, sino que limitó el modelo a una aproximación de rango $k$ con $k=10$ dimensiones. Esto es, redujeron todo el comportamiento a sólo 10 temáticas de Netflix.

Luego, se uso el teorema de Eckart-Young, que dice que la mejor aproximación de rango $k$ minimiza el error cuadrático medio bajo la norma de Frobenius. Esto se expresa como

$$\|W - U_k^T \Sigma_k V_k\|_F^2 = \sum_{ij} \left( w_{ij} - \sum_{k} \sigma_k u_{ki} v_{kj} \right)^2$$

Puesto que los datos de la competencia fueron muestreados de forma proporcional a la cantidad de calificaciones de cada usuario, los autores tuvieron que modificar la optimización para incluir la probabilidad $p_{ij}$ de que el par usuario-película fuera seleccionado, esto se enuncia por

$$\sum_{ij} p_{ij} \left( w_{ij} - \sum_{k} \sigma_k u_{ki} v_{kj} \right)^2$$

Para resolver el problema anterior, calcularon la SVD de la matriz escalada $\sqrt{p_{ij}} \cdot w_{ij}$ y luego dividieron el resultado punto a punto por $\sqrt{p_{ij}}$.

Adicionalmente, los métodos integrados fueron la predicción ingenua $P_{um}$, la correlación ítem a ítem, la aproximación SVD y las reglas de asociación.

La ecuación de predicción fue

$$ \text{Predicción} = 0.5533 P_{um} + 0.029 \text{ correlación} + 0.1987 \text{ SVD} - 0.0121 \text{ reglas\_asociación} - 0.0042 $$

Lo anterior, les permitió alcanzar un Root Mean Squared Error (RMSE) de $0.256$ y ganar el primer lugar de la competencia, además un hecho importante es que les tomó alrededor de 15 minutos utilizando el algoritmo de Lanczos en su clúster.

